In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Feedforward classico e flusso di controllo (circuiti dinamici)

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



# Feedforward classico e flusso di controllo


<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato con i seguenti requisiti.
Si consiglia di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
> **Note:** La nuova versione dei circuiti dinamici è ora disponibile per tutti gli utenti su tutti i backend. Puoi eseguire circuiti dinamici su scala applicativa. Consulta [l'annuncio](/announcements/product-updates/2025-09-25-new-dynamic-circuits) per ulteriori dettagli.

I circuiti dinamici sono strumenti potenti che ti permettono di misurare i qubit durante l'esecuzione di un circuito quantistico e di eseguire operazioni di logica classica all'interno del circuito, sulla base del risultato di quelle misure a metà circuito. Questo processo è noto anche come _feedforward classico_. Benché siamo ancora agli inizi della comprensione di come sfruttare al meglio i circuiti dinamici, la comunità di ricerca quantistica ha già identificato diversi casi d'uso, tra cui:

* Preparazione efficiente di stati quantistici, come lo [stato GHZ,](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) lo [stato W,](https://arxiv.org/abs/2403.07604) (per ulteriori informazioni sullo stato W, consulta anche ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)) e un'ampia classe di [stati prodotto a matrice](https://arxiv.org/abs/2404.16083)
* [Entanglement efficiente a lungo raggio](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) tra qubit sullo stesso chip tramite circuiti poco profondi
* [Campionamento efficiente di circuiti di tipo IQP](https://arxiv.org/pdf/2505.04705)

Questi miglioramenti apportati dai circuiti dinamici comportano, tuttavia, dei compromessi. Le misure a metà circuito e le operazioni classiche hanno tipicamente tempi di esecuzione più lunghi rispetto ai gate a due qubit, e questo aumento di tempo potrebbe vanificare i benefici della riduzione della profondità del circuito. Pertanto, la riduzione della durata delle misure a metà circuito è un'area di miglioramento prioritaria mentre IBM Quantum&reg; rilascia la [nuova versione](/announcements/product-updates/2025-03-03-new-version-dynamic-circuits) dei circuiti dinamici.

La [specifica OpenQASM 3](https://openqasm.com/language/classical.html#looping-and-branching) definisce diverse strutture di flusso di controllo, ma Qiskit Runtime attualmente supporta solo l'istruzione condizionale `if`. In Qiskit SDK, questo corrisponde al metodo [if_test](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test) su [QuantumCircuit.](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) Questo metodo restituisce un [context manager](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers) e viene tipicamente usato in un'istruzione `with`. Questa guida descrive come usare questa istruzione condizionale.

> **Note:** Gli esempi di codice in questa guida usano l'istruzione di misura standard per le misure a metà circuito. Tuttavia, si consiglia di usare l'istruzione [`MidCircuitMeasure`](/guides/measure-qubits#midcircuit) al suo posto, se il backend la supporta. Consulta la [documentazione sulle misure a metà circuito](/guides/measure-qubits#mid-circuit-measurements) per i dettagli.
## Istruzione `if`
L'istruzione `if` viene usata per eseguire operazioni in modo condizionale in base al valore di un bit o registro classico.

Nell'esempio seguente, applichiamo un gate di Hadamard a un qubit e lo misuriamo. Se il risultato è 1, applichiamo un gate X al qubit, che ha l'effetto di riportarlo allo stato 0. Poi misuriamo di nuovo il qubit. Il risultato della misura dovrebbe essere 0 con probabilità del 100%.

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg)

All'istruzione `with` può essere assegnato un target che è a sua volta un context manager, il quale può essere memorizzato e usato successivamente per creare un blocco else, eseguito ogni volta che il contenuto del blocco `if` *non* viene eseguito.

Nell'esempio seguente, inizializziamo dei registri con due qubit e due bit classici. Applichiamo un gate di Hadamard al primo qubit e lo misuriamo. Se il risultato è 1, applichiamo un gate di Hadamard al secondo qubit; altrimenti, applichiamo un gate X al secondo qubit. Infine, misuriamo anche il secondo qubit.

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg)

Oltre a condizionare su un singolo bit classico, è anche possibile condizionare sul valore di un registro classico composto da più bit.

Nell'esempio seguente, applichiamo gate di Hadamard a due qubit e li misuriamo. Se il risultato è `01`, ossia il primo qubit è 1 e il secondo qubit è 0, applichiamo un gate X a un terzo qubit. Infine, misuriamo il terzo qubit. Nota che per chiarezza abbiamo scelto di specificare lo stato del terzo bit classico, che è 0, nella condizione `if`. Nel disegno del circuito, la condizione è indicata dai cerchi sui bit classici su cui si condiziona. Un cerchio nero indica la condizione su 1, mentre un cerchio bianco indica la condizione su 0.

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg)

## Espressioni classiche
Il modulo di espressioni classiche di Qiskit [`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical) contiene una rappresentazione esplorativa delle operazioni runtime su valori classici durante l'esecuzione del circuito. A causa delle limitazioni hardware, attualmente sono supportate solo le condizioni di `QuantumCircuit.if_test()`.

L'esempio seguente mostra come usare il calcolo della parità per creare uno stato GHZ a n qubit usando i circuiti dinamici. Prima, si generano $n/2$ coppie di Bell su qubit adiacenti. Poi, si collegano queste coppie tramite un layer di gate CNOT tra le coppie. Si misurano quindi il qubit target di tutti i gate CNOT precedenti e si reimposta ogni qubit misurato allo stato $\vert 0 \rangle$. Si applica $X$ a ogni sito non misurato per il quale la parità di tutti i bit precedenti è dispari. Infine, i gate CNOT vengono applicati ai qubit misurati per ristabilire l'entanglement perso nella misura.

Nel calcolo della parità, il primo elemento dell'espressione costruita comporta il lifting dell'oggetto Python `mr[0]` a un nodo [`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value) (`lift` viene usato per convertire oggetti arbitrari in espressioni classiche). Questo non è necessario per `mr[1]` e il possibile registro classico successivo, poiché sono input di `expr.bit_xor`, e il lifting necessario viene eseguito automaticamente in questi casi. Tali espressioni possono essere costruite in cicli e altre strutture.

In [4]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [5]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg)

## Trovare i backend che supportano i circuiti dinamici
Per trovare tutti i backend a cui il tuo account può accedere e che supportano i circuiti dinamici, esegui un codice simile al seguente. Questo esempio presuppone che tu abbia [salvato le tue credenziali di accesso.](/guides/save-credentials) Puoi anche [specificare le credenziali esplicitamente](/guides/initialize-account#explicit) quando inizializzi il tuo account del servizio Qiskit Runtime. Questo ti permetterebbe di visualizzare i backend disponibili per una specifica istanza o tipo di piano, ad esempio.

> **Note:** - I backend disponibili per l'account dipendono dall'istanza specificata nelle credenziali.
> - La nuova versione dei circuiti dinamici è ora disponibile per tutti gli utenti su tutti i backend. Consulta [l'annuncio](/announcements/product-updates/2025-09-25-new-dynamic-circuits) per ulteriori dettagli.